# 01 — Coleta de dados de qualidade do ar e meteorologia

Este notebook consulta as APIs da **Open-Meteo** para o ponto de referência da Universidade Braz Cubas, em Mogi das Cruzes, e preserva as respostas originais em `data/raw/`.

## O que será feito

1. Ler localização, período, variáveis e caminhos de `config/params.yaml`.
2. Conferir os nomes dos arquivos e impedir que uma coleta substitua arquivos existentes.
3. Consultar a API de qualidade do ar e a API meteorológica.
4. Inspecionar a estrutura e a cobertura horária das respostas.
5. Salvar as respostas JSON sem limpeza ou transformação.

> **Antes de executar:** confirme o período em `config/params.yaml`. A execução faz chamadas externas e pode levar alguns instantes. As células de consulta e gravação não são executadas automaticamente ao abrir o notebook. Se os arquivos para o período já existirem, o notebook interromperá a execução sem sobrescrevê-los.

Execute as células em ordem, de cima para baixo, usando o ambiente que contém as dependências de `requirements.txt`.

## 1. Carregar as configurações do projeto

As coordenadas, datas, variáveis solicitadas, endereços das APIs e diretórios ficam centralizados no YAML. Assim, o código não precisa repetir esses parâmetros.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests
import yaml

ROOT = Path.cwd()
if not (ROOT / "config" / "params.yaml").is_file():
    ROOT = ROOT.parent
CONFIG_PATH = ROOT / "config" / "params.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Não encontrei a configuração do projeto: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as arquivo:
    config = yaml.safe_load(arquivo)

location = config["location"]
collection = config["collection"]
periodo = {
    "start_date": collection["start_date"],
    "end_date": collection["end_date"],
}
print(f"Ponto: {location['latitude']}, {location['longitude']} ({location['timezone']})")
print(f"Período configurado: {periodo['start_date']} a {periodo['end_date']}")
print(f"Configuração lida de: {CONFIG_PATH}")

## 2. Preparar os parâmetros e conferir os destinos

Cada fonte é salva em seu próprio JSON. A checagem acontece antes das consultas: se algum destino já existir, pare e confira o período ou arquive manualmente os arquivos anteriores. O notebook não apaga nem substitui dados.

In [ ]:
apis = config["apis"]
paths = config["paths"]
air_quality_config = apis["air_quality"]
weather_config = apis["weather"]

air_quality_path = ROOT / paths["raw_data"] / paths["air_quality_raw_filename"].format(**periodo)
weather_path = ROOT / paths["raw_data"] / paths["weather_raw_filename"].format(**periodo)
destinos_existentes = [str(path) for path in (air_quality_path, weather_path) if path.exists()]
if destinos_existentes:
    raise FileExistsError(
        "A coleta foi interrompida para não sobrescrever arquivo(s): "
        + ", ".join(destinos_existentes)
    )

params_air_quality = {
    "latitude": location["latitude"],
    "longitude": location["longitude"],
    "start_date": periodo["start_date"],
    "end_date": periodo["end_date"],
    "hourly": config["air_quality_variables"],
    "timezone": location["timezone"],
}
params_weather = {
    "latitude": location["latitude"],
    "longitude": location["longitude"],
    "start_date": periodo["start_date"],
    "end_date": periodo["end_date"],
    "hourly": config["weather_variables"],
    "timezone": location["timezone"],
}

print("Destinos disponíveis para esta coleta:")
print(f"- Qualidade do ar: {air_quality_path}")
print(f"- Meteorologia:    {weather_path}")
print(f"- Variáveis de qualidade do ar: {params_air_quality['hourly']}")
print(f"- Variáveis meteorológicas: {params_weather['hourly']}")

## 3. Consultar as duas APIs

`raise_for_status()` interrompe a execução quando o servidor responde com erro HTTP. O timeout vem da configuração de cada API. Nesta etapa, as respostas permanecem em memória; elas ainda não foram gravadas.

In [ ]:
response_air_quality = requests.get(
    air_quality_config["url"],
    params=params_air_quality,
    timeout=air_quality_config["timeout"],
)
response_air_quality.raise_for_status()
dados_air_quality = response_air_quality.json()

response_weather = requests.get(
    weather_config["url"],
    params=params_weather,
    timeout=weather_config["timeout"],
)
response_weather.raise_for_status()
dados_weather = response_weather.json()

print(f"Qualidade do ar: HTTP {response_air_quality.status_code}")
print(f"Meteorologia: HTTP {response_weather.status_code}")

## 4. Inspecionar formato e cobertura

A API devolve metadados e uma seção `hourly`. Esta inspeção confirma que há horários e que as séries possuem comprimentos compatíveis; não corrige nem preenche valores.

In [ ]:
def inspecionar_resposta(nome, resposta, variaveis_esperadas):
    hourly = resposta.get("hourly")
    if not isinstance(hourly, dict) or "time" not in hourly:
        raise ValueError(f"A resposta de {nome} não contém uma série 'hourly.time' válida.")

    comprimentos = {coluna: len(valores) for coluna, valores in hourly.items()}
    if len(set(comprimentos.values())) != 1:
        raise ValueError(f"Séries com comprimentos diferentes na resposta de {nome}: {comprimentos}")

    variaveis_ausentes = set(variaveis_esperadas).difference(hourly)
    if variaveis_ausentes:
        raise ValueError(f"Variáveis ausentes na resposta de {nome}: {sorted(variaveis_ausentes)}")

    horarios = pd.to_datetime(hourly["time"], errors="raise")
    print(f"{nome}: {len(horarios):,} horários")
    print(f"  Início: {horarios.min()}")
    print(f"  Fim:    {horarios.max()}")
    print(f"  Chaves da resposta: {list(resposta)}")
    print(f"  Variáveis horárias: {list(hourly)}")

inspecionar_resposta("Qualidade do ar", dados_air_quality, params_air_quality["hourly"])
inspecionar_resposta("Meteorologia", dados_weather, params_weather["hourly"])

## 5. Salvar os dados brutos

As respostas completas são mantidas em JSON, separadas por fonte. Nenhuma junção, limpeza, imputação ou cálculo do IQAr ocorre neste notebook. A célula verifica novamente os destinos imediatamente antes de gravar.

In [ ]:
destinos_existentes = [str(path) for path in (air_quality_path, weather_path) if path.exists()]
if destinos_existentes:
    raise FileExistsError(
        "A gravação foi interrompida para não sobrescrever arquivo(s): "
        + ", ".join(destinos_existentes)
    )

air_quality_path.parent.mkdir(parents=True, exist_ok=True)
weather_path.parent.mkdir(parents=True, exist_ok=True)
with air_quality_path.open("w", encoding="utf-8") as arquivo:
    json.dump(dados_air_quality, arquivo, ensure_ascii=False, indent=2)
with weather_path.open("w", encoding="utf-8") as arquivo:
    json.dump(dados_weather, arquivo, ensure_ascii=False, indent=2)

print("Coleta concluída. Arquivos brutos salvos:")
print(f"- {air_quality_path}")
print(f"- {weather_path}")
print("Os arquivos originais foram preservados sem transformações.")

## Próxima etapa

Após conferir os dois arquivos JSON em `data/raw/`, abra [`02_Merge_Dados.ipynb`](02_Merge_Dados.ipynb) para inspecionar as séries e integrá-las pelo horário. A alternativa sem interface de notebook continua disponível em `src/coleta/Coleta_Dados.py`.